In [1]:
# ============================================================
# Cell 1 — Load from HuggingFace, 6-season dataset
# ============================================================
!pip install -q huggingface_hub

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
import pandas as pd
import numpy as np

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("200205")
login(token=HF_TOKEN)

REPO_ID = "Jahid05/wildfire-early-warning-sensors-v2"  # your 6-season repo

query_points = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="query_points.csv", repo_type="dataset", token=HF_TOKEN))
weather_df = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="weather_raw.csv", repo_type="dataset", token=HF_TOKEN))
openaq_df = pd.read_csv(hf_hub_download(repo_id=REPO_ID, filename="openaq_raw.csv", repo_type="dataset", token=HF_TOKEN))

query_points["ref_time"] = pd.to_datetime(query_points["ref_time"], utc=True)
weather_df["time"] = pd.to_datetime(weather_df["time"], utc=True)
openaq_df["datetime"] = pd.to_datetime(openaq_df["datetime"], utc=True)

HORIZONS_HOURS = [6, 12, 24, 48, 72]
FEATURE_WINDOW_HOURS = 48

print(f"{len(query_points)} query points, {len(HORIZONS_HOURS)} horizons -> {len(query_points)*len(HORIZONS_HOURS)} (point, horizon) pairs")
print(f"weather_df dtype: {weather_df['time'].dtype}, openaq_df dtype: {openaq_df['datetime'].dtype}")
# Expect 904 query points -> 4520 (point, horizon) rows, roughly 2.8x your original 322/1610

query_points.csv: 0.00B [00:00, ?B/s]

weather_raw.csv: 0.00B [00:00, ?B/s]

openaq_raw.csv: 0.00B [00:00, ?B/s]

904 query points, 5 horizons -> 4520 (point, horizon) pairs
weather_df dtype: datetime64[ns, UTC], openaq_df dtype: datetime64[ns, UTC]


In [2]:
# ============================================================
# Cell 2 — Leakage-safe extraction functions (unchanged)
# ============================================================
def extract_weather_features(qp_idx, cutoff, weather_df, window_hours=FEATURE_WINDOW_HOURS):
    window_start = cutoff - pd.Timedelta(hours=window_hours)
    subset = weather_df[
        (weather_df["query_point_idx"] == qp_idx) &
        (weather_df["time"] < cutoff) &
        (weather_df["time"] >= window_start)
    ]
    if len(subset) == 0:
        return None

    feats = {}
    for col in ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "precipitation"]:
        feats[f"weather_{col}_mean"] = subset[col].mean()
        feats[f"weather_{col}_std"] = subset[col].std()
        feats[f"weather_{col}_max"] = subset[col].max()
        feats[f"weather_{col}_min"] = subset[col].min()
        feats[f"weather_{col}_last"] = subset.sort_values("time")[col].iloc[-1]
    feats["weather_n_obs"] = len(subset)
    return feats

def extract_openaq_features(qp_idx, cutoff, openaq_df, window_hours=FEATURE_WINDOW_HOURS):
    window_start = cutoff - pd.Timedelta(hours=window_hours)
    subset = openaq_df[
        (openaq_df["query_point_idx"] == qp_idx) &
        (openaq_df["datetime"] < cutoff) &
        (openaq_df["datetime"] >= window_start)
    ]
    if len(subset) == 0:
        return None

    feats = {}
    for param in subset["parameter"].unique():
        param_data = subset[subset["parameter"] == param]["value"]
        feats[f"openaq_{param}_mean"] = param_data.mean()
        feats[f"openaq_{param}_max"] = param_data.max()
        feats[f"openaq_{param}_last"] = param_data.iloc[-1]
    feats["openaq_n_obs"] = len(subset)
    return feats

print("Extraction functions defined.")

Extraction functions defined.


In [3]:
# ============================================================
# Cell 3 — Build the full feature table
# ============================================================
rows = []

for idx, row in query_points.iterrows():
    for horizon in HORIZONS_HOURS:
        cutoff = row["ref_time"] - pd.Timedelta(hours=horizon)

        weather_feats = extract_weather_features(idx, cutoff, weather_df)
        openaq_feats = extract_openaq_features(idx, cutoff, openaq_df)

        record = {
            "query_point_idx": idx,
            "horizon_hours": horizon,
            "cutoff_time": cutoff,
            "label": 1 if row["point_type"] == "positive" else 0,
            "split": row["split"],
            "has_weather": weather_feats is not None,
            "has_openaq": openaq_feats is not None,
        }
        if weather_feats:
            record.update(weather_feats)
        if openaq_feats:
            record.update(openaq_feats)

        rows.append(record)

    if idx % 100 == 0:
        print(f"  {idx}/{len(query_points)} points processed")

features_df = pd.DataFrame(rows)
print(f"\nTotal (point, horizon) rows: {len(features_df)}")
print(f"Weather availability: {features_df['has_weather'].mean():.1%}")
print(f"OpenAQ availability: {features_df['has_openaq'].mean():.1%}")
print("\nOpenAQ availability by horizon:")
print(features_df.groupby("horizon_hours")["has_openaq"].mean())

  0/904 points processed
  100/904 points processed
  200/904 points processed
  300/904 points processed
  400/904 points processed
  500/904 points processed
  600/904 points processed
  700/904 points processed
  800/904 points processed
  900/904 points processed

Total (point, horizon) rows: 4520
Weather availability: 99.8%
OpenAQ availability: 25.5%

OpenAQ availability by horizon:
horizon_hours
6     0.255531
12    0.256637
24    0.256637
48    0.253319
72    0.251106
Name: has_openaq, dtype: float64


In [4]:
# ============================================================
# Cell 4 — Leakage sanity check (re-verify against the new 6-season data)
# ============================================================
sample = query_points[query_points["point_type"]=="positive"].iloc[0]
sample_idx = sample.name
horizon = 6
cutoff = sample["ref_time"] - pd.Timedelta(hours=horizon)

leaked = weather_df[(weather_df["query_point_idx"] == sample_idx) & (weather_df["time"] >= cutoff)]
print(f"ref_time: {sample['ref_time']}, cutoff (T-{horizon}h): {cutoff}")
print(f"Rows at/after cutoff that MUST be excluded: {len(leaked)}")

check_row = features_df[(features_df["query_point_idx"]==sample_idx) & (features_df["horizon_hours"]==horizon)]
print(check_row[["cutoff_time", "weather_n_obs"]])

openaq_sample_idx = openaq_df["query_point_idx"].iloc[0]
oaq_row = query_points.loc[openaq_sample_idx]
oaq_cutoff = oaq_row["ref_time"] - pd.Timedelta(hours=horizon)
oaq_leaked = openaq_df[(openaq_df["query_point_idx"] == openaq_sample_idx) & (openaq_df["datetime"] >= oaq_cutoff)]
print(f"\nOpenAQ check — point {openaq_sample_idx}, cutoff: {oaq_cutoff}")
print(f"Rows at/after cutoff that MUST be excluded: {len(oaq_leaked)}")

ref_time: 2019-08-01 10:20:00+00:00, cutoff (T-6h): 2019-08-01 04:20:00+00:00
Rows at/after cutoff that MUST be excluded: 19
                cutoff_time  weather_n_obs
0 2019-08-01 04:20:00+00:00           48.0

OpenAQ check — point 5, cutoff: 2023-09-07 04:34:00+00:00
Rows at/after cutoff that MUST be excluded: 2


In [5]:
# ============================================================
# Cell 5 — Save and upload
# ============================================================
import os
os.makedirs("/kaggle/working/raw", exist_ok=True)
features_df.to_csv("/kaggle/working/raw/features_all_horizons.csv", index=False)

from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="/kaggle/working/raw/features_all_horizons.csv",
    path_in_repo="features_all_horizons.csv",
    repo_id=REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN
)
print("features_all_horizons.csv uploaded.")

features_all_horizons.csv uploaded.


In [6]:
# ============================================================
# Build the five arms (same logic as before, new data)
# ============================================================
base_cols = {"query_point_idx", "horizon_hours", "cutoff_time", "label", "split"}
weather_cols = [c for c in features_df.columns if c.startswith("weather_")]
openaq_cols = [c for c in features_df.columns if c.startswith("openaq_")]

arm_weather_only = features_df[list(base_cols) + weather_cols].copy()
arm_openaq_only = features_df[features_df["has_openaq"] == True][list(base_cols) + openaq_cols].copy()
arm_combined = features_df[
    (features_df["has_weather"] == True) & (features_df["has_openaq"] == True)
][list(base_cols) + weather_cols + openaq_cols].copy()
arm_missingness_aware = features_df[list(base_cols) + ["has_weather", "has_openaq"] + weather_cols + openaq_cols].copy()

matched_indices = arm_combined[["query_point_idx", "horizon_hours"]]
arm_weather_matched = arm_weather_only.merge(matched_indices, on=["query_point_idx", "horizon_hours"], how="inner")

arms = {
    "weather_only": arm_weather_only,
    "weather_matched": arm_weather_matched,
    "combined": arm_combined,
    "openaq_only": arm_openaq_only,
    "missingness_aware": arm_missingness_aware,
}

for name, df in arms.items():
    print(f"{name}: {len(df)} rows")

weather_only: 4520 rows
weather_matched: 1146 rows
combined: 1146 rows
openaq_only: 1151 rows
missingness_aware: 4520 rows


In [7]:
# ============================================================
# Spot check: raw missingness fraction per OpenAQ parameter column,
# BEFORE applying the 50% threshold — see where things actually land now
# ============================================================
for arm_name in ["combined", "openaq_only"]:
    df = arms[arm_name]
    oaq_feature_cols = [c for c in df.columns if c.startswith("openaq_")]
    nan_frac = df[oaq_feature_cols].isna().mean().sort_values()
    print(f"\n=== {arm_name} — OpenAQ column missingness (sorted, ascending) ===")
    print(nan_frac)
    print(f"\nColumns that would SURVIVE a 50% threshold: {(nan_frac <= 0.5).sum()}/{len(nan_frac)}")
    print(f"Columns that would be DROPPED: {(nan_frac > 0.5).sum()}/{len(nan_frac)}")


=== combined — OpenAQ column missingness (sorted, ascending) ===
openaq_n_obs                    0.000000
openaq_o3_mean                  0.308901
openaq_o3_max                   0.308901
openaq_o3_last                  0.308901
openaq_pm25_mean                0.397033
openaq_pm25_max                 0.397033
openaq_pm25_last                0.424084
openaq_no2_mean                 0.839442
openaq_no2_max                  0.839442
openaq_no2_last                 0.839442
openaq_pm10_mean                0.908377
openaq_pm10_max                 0.908377
openaq_pm10_last                0.909250
openaq_no_mean                  0.947644
openaq_no_max                   0.947644
openaq_nox_mean                 0.947644
openaq_nox_last                 0.947644
openaq_nox_max                  0.947644
openaq_no_last                  0.949389
openaq_co_mean                  0.952007
openaq_co_max                   0.952007
openaq_co_last                  0.952007
openaq_bc_max                   

In [8]:
threshold = 0.5
combined_oaq_missingness = arms["combined"][[c for c in arms["combined"].columns if c.startswith("openaq_")]].isna().mean()
kept = combined_oaq_missingness[combined_oaq_missingness <= threshold].index.tolist()
print(f"Columns surviving 50% threshold: {kept}")

Columns surviving 50% threshold: ['openaq_o3_mean', 'openaq_o3_max', 'openaq_o3_last', 'openaq_n_obs', 'openaq_pm25_mean', 'openaq_pm25_max', 'openaq_pm25_last']


In [9]:
def drop_sparse_columns(df, base_cols, threshold=0.5):
    feature_cols = [c for c in df.columns if c not in base_cols]
    nan_frac = df[feature_cols].isna().mean()
    keep_cols = nan_frac[nan_frac <= threshold].index.tolist()
    dropped_cols = nan_frac[nan_frac > threshold].index.tolist()
    if dropped_cols:
        print(f"  dropped {len(dropped_cols)} sparse columns")
    return df[list(base_cols) + keep_cols], keep_cols

def drop_sparse_columns_conditional(df, base_cols, threshold=0.5, condition_col=None):
    feature_cols = [c for c in df.columns if c not in base_cols]
    relevant_rows = df[df[condition_col] == True] if condition_col else df
    nan_frac = relevant_rows[feature_cols].isna().mean()
    keep_cols = nan_frac[nan_frac <= threshold].index.tolist()
    dropped_cols = nan_frac[nan_frac > threshold].index.tolist()
    if dropped_cols:
        print(f"  dropped {len(dropped_cols)} sparse columns")
    return df[list(base_cols) + keep_cols], keep_cols

base_cols_set = base_cols
cleaned_arms = {}

for name, df in arms.items():
    if name == "missingness_aware":
        cleaned_df, kept = drop_sparse_columns_conditional(df, base_cols_set, threshold=0.5, condition_col="has_openaq")
    else:
        cleaned_df, kept = drop_sparse_columns(df, base_cols_set, threshold=0.5)
    cleaned_arms[name] = cleaned_df
    print(f"{name}: {len(kept)} features retained\n")

weather_only: 21 features retained

weather_matched: 21 features retained

  dropped 33 sparse columns
combined: 28 features retained

  dropped 33 sparse columns
openaq_only: 7 features retained

  dropped 33 sparse columns
missingness_aware: 30 features retained



In [10]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score

def prepare_arm_horizon(df, horizon, feature_cols):
    subset = df[df["horizon_hours"] == horizon].copy()
    train = subset[subset["split"] == "train"]
    test = subset[subset["split"] == "test"]

    X_train_raw = train[feature_cols].values
    X_test_raw = test[feature_cols].values
    y_train = train["label"].values
    y_test = test["label"].values

    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp = imputer.transform(X_test_raw)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)

    return X_train_imp, X_test_imp, X_train_scaled, X_test_scaled, y_train, y_test

def bootstrap_pr_auc(y_true, y_scores, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        y_b, s_b = y_true[idx], y_scores[idx]
        if len(np.unique(y_b)) < 2:
            continue
        scores.append(average_precision_score(y_b, s_b))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

HORIZONS_HOURS = [6, 12, 24, 48, 72]
results = []

for arm_name, df in cleaned_arms.items():
    feature_cols = [c for c in df.columns if c not in base_cols_set]
    print(f"\n=== Arm: {arm_name} ({len(feature_cols)} features) ===")

    for horizon in HORIZONS_HOURS:
        X_train_imp, X_test_imp, X_train_s, X_test_s, y_train, y_test = prepare_arm_horizon(df, horizon, feature_cols)

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(f"  horizon {horizon}h: skipped, single-class split")
            continue

        pos_rate_test = y_test.mean()

        lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
        lr.fit(X_train_s, y_train)
        lr_scores = lr.predict_proba(X_test_s)[:, 1]
        lr_mean, lr_lo, lr_hi = bootstrap_pr_auc(y_test, lr_scores)

        xgb = XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
            eval_metric="logloss", random_state=42
        )
        xgb.fit(X_train_imp, y_train)
        xgb_scores = xgb.predict_proba(X_test_imp)[:, 1]
        xgb_mean, xgb_lo, xgb_hi = bootstrap_pr_auc(y_test, xgb_scores)

        results.append({"arm": arm_name, "horizon_hours": horizon, "model": "logistic_regression",
                         "pr_auc": lr_mean, "ci_low": lr_lo, "ci_high": lr_hi,
                         "n_train": len(y_train), "n_test": len(y_test), "test_pos_rate": pos_rate_test})
        results.append({"arm": arm_name, "horizon_hours": horizon, "model": "xgboost",
                         "pr_auc": xgb_mean, "ci_low": xgb_lo, "ci_high": xgb_hi,
                         "n_train": len(y_train), "n_test": len(y_test), "test_pos_rate": pos_rate_test})

        print(f"  horizon {horizon}h: LR PR-AUC={lr_mean:.3f} [{lr_lo:.3f},{lr_hi:.3f}] | XGB PR-AUC={xgb_mean:.3f} [{xgb_lo:.3f},{xgb_hi:.3f}]")

results_df = pd.DataFrame(results)
results_df["chance_level"] = results_df["test_pos_rate"]
results_df["lift_over_chance"] = results_df["pr_auc"] / results_df["chance_level"]

import os
os.makedirs("/kaggle/working/raw", exist_ok=True)
results_df.to_csv("/kaggle/working/raw/phase3_baseline_results_6season.csv", index=False)
print(f"\nSaved {len(results_df)} result rows.")


=== Arm: weather_only (21 features) ===
  horizon 6h: LR PR-AUC=0.862 [0.779,0.930] | XGB PR-AUC=0.892 [0.836,0.938]
  horizon 12h: LR PR-AUC=0.853 [0.762,0.929] | XGB PR-AUC=0.892 [0.835,0.938]
  horizon 24h: LR PR-AUC=0.889 [0.832,0.935] | XGB PR-AUC=0.886 [0.821,0.936]
  horizon 48h: LR PR-AUC=0.899 [0.843,0.941] | XGB PR-AUC=0.901 [0.841,0.946]
  horizon 72h: LR PR-AUC=0.910 [0.861,0.950] | XGB PR-AUC=0.890 [0.828,0.937]

=== Arm: weather_matched (21 features) ===
  horizon 6h: LR PR-AUC=0.977 [0.942,0.998] | XGB PR-AUC=0.941 [0.870,0.983]
  horizon 12h: LR PR-AUC=0.858 [0.706,0.970] | XGB PR-AUC=0.870 [0.720,0.975]
  horizon 24h: LR PR-AUC=0.926 [0.844,0.983] | XGB PR-AUC=0.932 [0.853,0.985]
  horizon 48h: LR PR-AUC=0.943 [0.869,0.987] | XGB PR-AUC=0.944 [0.876,0.987]
  horizon 72h: LR PR-AUC=0.932 [0.861,0.978] | XGB PR-AUC=0.907 [0.793,0.982]

=== Arm: combined (28 features) ===
  horizon 6h: LR PR-AUC=0.965 [0.916,1.000] | XGB PR-AUC=0.950 [0.884,0.993]
  horizon 12h: LR PR-AU

In [11]:
pivot = results_df.pivot_table(index=["arm","horizon_hours"], columns="model", values="lift_over_chance")
print(pivot)

model                            logistic_regression   xgboost
arm               horizon_hours                               
combined          6                         1.338777  1.318052
                  12                        1.193761  1.251881
                  24                        1.301016  1.270823
                  48                        1.319627  1.319670
                  72                        1.310911  1.240404
missingness_aware 6                         1.732923  1.817879
                  12                        1.719891  1.786531
                  24                        1.802667  1.778561
                  48                        1.829646  1.811895
                  72                        1.817289  1.748849
openaq_only       6                         1.166298  1.276206
                  12                        1.231625  1.200154
                  24                        1.198276  1.207458
                  48                        1.213567  1

In [12]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

SEQ_LENGTH = 48  # hours of history per sequence, matches Phase 2's feature window
WEATHER_VARS = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "precipitation"]

def build_sequence(qp_idx, cutoff, weather_df, seq_hours=SEQ_LENGTH):
    window_start = cutoff - pd.Timedelta(hours=seq_hours)
    subset = weather_df[
        (weather_df["query_point_idx"] == qp_idx) &
        (weather_df["time"] < cutoff) &          # same leakage-safe boundary as Phase 2
        (weather_df["time"] >= window_start)
    ].sort_values("time")

    if len(subset) == 0:
        return None

    # Build a full hourly grid, mark which hours are actually observed (mask)
    full_grid = pd.date_range(window_start, cutoff, freq="h", inclusive="left")
    grid_df = pd.DataFrame({"time": full_grid})
    merged = grid_df.merge(subset[["time"] + WEATHER_VARS], on="time", how="left")

    values = merged[WEATHER_VARS].values                     # (seq_len, n_vars), NaN where missing
    mask = (~merged[WEATHER_VARS].isna()).values.astype(float)  # 1 = observed, 0 = missing
    # time since last observation, per variable (GRU-D's delta term)
    deltas = np.zeros_like(values, dtype=float)
    for v in range(values.shape[1]):
        last_obs = 0
        for t in range(len(values)):
            if mask[t, v] == 1:
                deltas[t, v] = 0
                last_obs = t
            else:
                deltas[t, v] = t - last_obs if t > 0 else 0

    return values, mask, deltas

print("Sequence builder defined.")

Sequence builder defined.


In [13]:
def build_horizon_dataset(query_points, weather_df, horizon):
    X_list, M_list, D_list, y_list, split_list = [], [], [], [], []

    for idx, row in query_points.iterrows():
        cutoff = row["ref_time"] - pd.Timedelta(hours=horizon)
        result = build_sequence(idx, cutoff, weather_df)
        if result is None:
            continue
        values, mask, deltas = result

        # forward-fill NaN for the "last observed value" GRU-D needs; keep mask separate as the truth signal
        values_filled = pd.DataFrame(values).ffill().bfill().fillna(0).values

        X_list.append(values_filled)
        M_list.append(mask)
        D_list.append(deltas)
        y_list.append(1 if row["point_type"] == "positive" else 0)
        split_list.append(row["split"])

    return (np.array(X_list), np.array(M_list), np.array(D_list),
            np.array(y_list), np.array(split_list))

print("Horizon dataset builder defined.")

Horizon dataset builder defined.


In [14]:
class GRUD(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.input_size = input_size

        # decay for hidden state and input, based on time-since-observed
        self.gamma_x = nn.Linear(input_size, input_size)
        self.gamma_h = nn.Linear(input_size, hidden_size)

        self.gru_cell = nn.GRUCell(input_size * 2, hidden_size)  # input + mask concatenated
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, X, M, D):
        # X, M, D: (batch, seq_len, input_size)
        batch_size, seq_len, _ = X.shape
        h = torch.zeros(batch_size, self.hidden_size, device=X.device)

        x_last_obs = X[:, 0, :].clone()

        for t in range(seq_len):
            x_t, m_t, d_t = X[:, t, :], M[:, t, :], D[:, t, :]

            gamma_x_t = torch.exp(-torch.relu(self.gamma_x(d_t)))
            x_decayed = gamma_x_t * x_last_obs + (1 - gamma_x_t) * x_t
            x_last_obs = torch.where(m_t.bool(), x_t, x_decayed)

            gamma_h_t = torch.exp(-torch.relu(self.gamma_h(d_t)))
            h = gamma_h_t * h

            gru_input = torch.cat([x_decayed, m_t], dim=1)
            h = self.gru_cell(gru_input, h)

        return self.classifier(h).squeeze(-1)

print("GRU-D model defined.")

GRU-D model defined.


In [15]:
from sklearn.metrics import average_precision_score

def bootstrap_pr_auc(y_true, y_scores, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    scores = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        y_b, s_b = y_true[idx], y_scores[idx]
        if len(np.unique(y_b)) < 2:
            continue
        scores.append(average_precision_score(y_b, s_b))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

def train_grud_for_horizon(query_points, weather_df, horizon, epochs=30, device="cuda" if torch.cuda.is_available() else "cpu"):
    X, M, D, y, split = build_horizon_dataset(query_points, weather_df, horizon)

    train_mask = split == "train"
    test_mask = split == "test"

    X_train = torch.tensor(X[train_mask], dtype=torch.float32).to(device)
    M_train = torch.tensor(M[train_mask], dtype=torch.float32).to(device)
    D_train = torch.tensor(D[train_mask], dtype=torch.float32).to(device)
    y_train = torch.tensor(y[train_mask], dtype=torch.float32).to(device)

    X_test = torch.tensor(X[test_mask], dtype=torch.float32).to(device)
    M_test = torch.tensor(M[test_mask], dtype=torch.float32).to(device)
    D_test = torch.tensor(D[test_mask], dtype=torch.float32).to(device)
    y_test_np = y[test_mask]

    model = GRUD(input_size=X.shape[2], hidden_size=32).to(device)
    pos_weight = torch.tensor([(y_train == 0).sum() / max((y_train == 1).sum(), 1)]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train, M_train, D_train)
        loss = criterion(logits, y_train)
        loss.backward()
        optimizer.step()
        if epoch % 10 == 0:
            print(f"    epoch {epoch}: loss={loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test, M_test, D_test)
        test_scores = torch.sigmoid(test_logits).cpu().numpy()

    pr_mean, pr_lo, pr_hi = bootstrap_pr_auc(y_test_np, test_scores)
    return pr_mean, pr_lo, pr_hi, len(y_train), len(y_test_np), y_test_np.mean()

HORIZONS_HOURS = [6, 12, 24, 48, 72]
grud_results = []

for horizon in HORIZONS_HOURS:
    print(f"\n=== GRU-D, horizon {horizon}h ===")
    pr_mean, pr_lo, pr_hi, n_train, n_test, pos_rate = train_grud_for_horizon(query_points, weather_df, horizon)
    grud_results.append({"arm": "grud_weather", "horizon_hours": horizon, "model": "grud",
                          "pr_auc": pr_mean, "ci_low": pr_lo, "ci_high": pr_hi,
                          "n_train": n_train, "n_test": n_test, "test_pos_rate": pos_rate,
                          "chance_level": pos_rate, "lift_over_chance": pr_mean/pos_rate})
    print(f"  PR-AUC={pr_mean:.3f} [{pr_lo:.3f},{pr_hi:.3f}]")

grud_results_df = pd.DataFrame(grud_results)
grud_results_df.to_csv("/kaggle/working/raw/phase4_grud_results.csv", index=False)
print(grud_results_df)


=== GRU-D, horizon 6h ===
    epoch 0: loss=0.5368
    epoch 10: loss=0.4263
    epoch 20: loss=0.3617
  PR-AUC=1.000 [1.000,1.000]

=== GRU-D, horizon 12h ===
    epoch 0: loss=0.7004
    epoch 10: loss=0.6198
    epoch 20: loss=0.5498
  PR-AUC=1.000 [1.000,1.000]

=== GRU-D, horizon 24h ===
    epoch 0: loss=0.6327
    epoch 10: loss=0.5548
    epoch 20: loss=0.4837
  PR-AUC=1.000 [1.000,1.000]

=== GRU-D, horizon 48h ===
    epoch 0: loss=0.7679
    epoch 10: loss=0.6287
    epoch 20: loss=0.5221
  PR-AUC=1.000 [1.000,1.000]

=== GRU-D, horizon 72h ===
    epoch 0: loss=0.7321
    epoch 10: loss=0.5866
    epoch 20: loss=0.4736
  PR-AUC=1.000 [1.000,1.000]
            arm  horizon_hours model  pr_auc  ci_low  ci_high  n_train  \
0  grud_weather              6  grud     1.0     1.0      1.0      731   
1  grud_weather             12  grud     1.0     1.0      1.0      731   
2  grud_weather             24  grud     1.0     1.0      1.0      731   
3  grud_weather             48  gru

In [16]:
# Check 1: are any test sequences literally identical between classes (data duplication bug)?
X, M, D, y, split = build_horizon_dataset(query_points, weather_df, 6)
test_mask = split == "test"
X_test, y_test = X[test_mask], y[test_mask]

print(f"Unique sequences in test set: {len(np.unique(X_test.reshape(len(X_test), -1), axis=0))} / {len(X_test)}")

# Check 2: does the model even need to look at the data, or is there a trivial separator?
# e.g., check if mask patterns alone (not values) perfectly separate the classes
M_test = M[test_mask]
mask_sums = M_test.sum(axis=(1,2))  # total observed count per sequence
import scipy.stats as stats
print(stats.pointbiserialr(y_test, mask_sums))

Unique sequences in test set: 86 / 171
SignificanceResult(statistic=np.float64(-1.0), pvalue=np.float64(0.0))


In [17]:
# How many test sequences are ENTIRELY missing (all mask=0)?
fully_missing = (M_test.sum(axis=(1,2)) == 0)
print(f"Fully-missing sequences: {fully_missing.sum()}/{len(M_test)}")
print(f"Of those, class distribution: {y_test[fully_missing]}")

Fully-missing sequences: 86/171
Of those, class distribution: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1]


In [18]:
# Verify exactly where the leak enters — check whether these are truly zero-row subsets,
# or subsets with the SAME rows across all fully-missing sequences (a different bug)
qp_idx_positive_sample = query_points[(query_points["split"]=="test") & (query_points["point_type"]=="positive")].index[0]
cutoff = query_points.loc[qp_idx_positive_sample, "ref_time"] - pd.Timedelta(hours=6)
window_start = cutoff - pd.Timedelta(hours=48)

subset_check = weather_df[
    (weather_df["query_point_idx"] == qp_idx_positive_sample) &
    (weather_df["time"] < cutoff) &
    (weather_df["time"] >= window_start)
]
print(f"query_point_idx: {qp_idx_positive_sample}, subset rows: {len(subset_check)}")

# Cross-check: does weather_df even USE this exact query_point_idx numbering?
print(f"weather_df query_point_idx range: {weather_df['query_point_idx'].min()} to {weather_df['query_point_idx'].max()}")
print(f"query_points index range: {query_points.index.min()} to {query_points.index.max()}")
print(f"weather_df unique query_point_idx count: {weather_df['query_point_idx'].nunique()}")

query_point_idx: 49, subset rows: 48
weather_df query_point_idx range: 0 to 903
query_points index range: 0 to 903
weather_df unique query_point_idx count: 902


In [19]:
def build_sequence(qp_idx, cutoff, weather_df, seq_hours=SEQ_LENGTH):
    window_start = cutoff - pd.Timedelta(hours=seq_hours)

    # Leakage-safe filter — exact cutoff, unchanged, unrounded
    subset = weather_df[
        (weather_df["query_point_idx"] == qp_idx) &
        (weather_df["time"] < cutoff) &
        (weather_df["time"] >= window_start)
    ].sort_values("time")

    if len(subset) == 0:
        return None

    # Grid must align to weather_df's actual hourly timestamps (floor to the hour),
    # NOT to cutoff's raw minute value — this is what was broken
    grid_end = cutoff.floor("h")
    grid_start = grid_end - pd.Timedelta(hours=seq_hours)
    full_grid = pd.date_range(grid_start, grid_end, freq="h", inclusive="left")

    grid_df = pd.DataFrame({"time": full_grid})
    merged = grid_df.merge(subset[["time"] + WEATHER_VARS], on="time", how="left")

    values = merged[WEATHER_VARS].values
    mask = (~merged[WEATHER_VARS].isna()).values.astype(float)

    deltas = np.zeros_like(values, dtype=float)
    for v in range(values.shape[1]):
        last_obs = 0
        for t in range(len(values)):
            if mask[t, v] == 1:
                deltas[t, v] = 0
                last_obs = t
            else:
                deltas[t, v] = t - last_obs if t > 0 else 0

    return values, mask, deltas

In [20]:
X, M, D, y, split = build_horizon_dataset(query_points, weather_df, 6)
test_mask = split == "test"
M_test, y_test = M[test_mask], y[test_mask]

fully_missing = (M_test.sum(axis=(1,2)) == 0)
print(f"Fully-missing sequences: {fully_missing.sum()}/{len(M_test)}")
if fully_missing.sum() > 0:
    print(f"Class distribution among fully-missing: {np.bincount(y_test[fully_missing].astype(int))}")

Fully-missing sequences: 0/171


In [21]:
grud_results = []

for horizon in HORIZONS_HOURS:
    print(f"\n=== GRU-D, horizon {horizon}h ===")
    pr_mean, pr_lo, pr_hi, n_train, n_test, pos_rate = train_grud_for_horizon(query_points, weather_df, horizon)
    grud_results.append({"arm": "grud_weather", "horizon_hours": horizon, "model": "grud",
                          "pr_auc": pr_mean, "ci_low": pr_lo, "ci_high": pr_hi,
                          "n_train": n_train, "n_test": n_test, "test_pos_rate": pos_rate,
                          "chance_level": pos_rate, "lift_over_chance": pr_mean/pos_rate})
    print(f"  PR-AUC={pr_mean:.3f} [{pr_lo:.3f},{pr_hi:.3f}]")

grud_results_df = pd.DataFrame(grud_results)
grud_results_df.to_csv("/kaggle/working/raw/phase4_grud_results.csv", index=False)
print(grud_results_df)


=== GRU-D, horizon 6h ===
    epoch 0: loss=0.7123
    epoch 10: loss=0.6683
    epoch 20: loss=0.6406
  PR-AUC=0.773 [0.692,0.848]

=== GRU-D, horizon 12h ===
    epoch 0: loss=0.7184
    epoch 10: loss=0.6649
    epoch 20: loss=0.6352
  PR-AUC=0.794 [0.713,0.865]

=== GRU-D, horizon 24h ===
    epoch 0: loss=0.6869
    epoch 10: loss=0.6550
    epoch 20: loss=0.6347
  PR-AUC=0.845 [0.764,0.915]

=== GRU-D, horizon 48h ===
    epoch 0: loss=0.7377
    epoch 10: loss=0.6903
    epoch 20: loss=0.6641
  PR-AUC=0.819 [0.737,0.888]

=== GRU-D, horizon 72h ===
    epoch 0: loss=0.7214
    epoch 10: loss=0.6605
    epoch 20: loss=0.6345
  PR-AUC=0.864 [0.779,0.929]
            arm  horizon_hours model    pr_auc    ci_low   ci_high  n_train  \
0  grud_weather              6  grud  0.773090  0.691517  0.847752      731   
1  grud_weather             12  grud  0.793588  0.713458  0.864533      731   
2  grud_weather             24  grud  0.844660  0.764143  0.914648      731   
3  grud_weather

In [22]:
for horizon in HORIZONS_HOURS:
    X, M, D, y, split = build_horizon_dataset(query_points, weather_df, horizon)
    test_mask = split == "test"
    M_test, y_test = M[test_mask], y[test_mask]
    fully_missing = (M_test.sum(axis=(1,2)) == 0)
    mask_sums = M_test.sum(axis=(1,2))
    corr = stats.pointbiserialr(y_test, mask_sums)
    print(f"horizon {horizon}h: fully_missing={fully_missing.sum()}/{len(M_test)}, mask-label correlation={corr.statistic:.3f} (p={corr.pvalue:.3f})")

horizon 6h: fully_missing=0/171, mask-label correlation=-1.000 (p=0.000)
horizon 12h: fully_missing=0/171, mask-label correlation=-1.000 (p=0.000)
horizon 24h: fully_missing=0/171, mask-label correlation=-1.000 (p=0.000)
horizon 48h: fully_missing=0/171, mask-label correlation=-1.000 (p=0.000)
horizon 72h: fully_missing=0/171, mask-label correlation=-1.000 (p=0.000)


In [23]:
def build_sequence(qp_idx, cutoff, weather_df, seq_hours=SEQ_LENGTH):
    # Floor the cutoff itself, applied consistently to BOTH the filter and the grid —
    # this is strictly more conservative than the raw cutoff (floor only rounds down),
    # so leakage-safety is preserved, and now positives/negatives share the same reference frame
    cutoff_floor = cutoff.floor("h")
    window_start = cutoff_floor - pd.Timedelta(hours=seq_hours)

    subset = weather_df[
        (weather_df["query_point_idx"] == qp_idx) &
        (weather_df["time"] < cutoff_floor) &      # floored, matches grid exactly now
        (weather_df["time"] >= window_start)
    ].sort_values("time")

    if len(subset) == 0:
        return None

    full_grid = pd.date_range(window_start, cutoff_floor, freq="h", inclusive="left")
    grid_df = pd.DataFrame({"time": full_grid})
    merged = grid_df.merge(subset[["time"] + WEATHER_VARS], on="time", how="left")

    values = merged[WEATHER_VARS].values
    mask = (~merged[WEATHER_VARS].isna()).values.astype(float)

    deltas = np.zeros_like(values, dtype=float)
    for v in range(values.shape[1]):
        last_obs = 0
        for t in range(len(values)):
            if mask[t, v] == 1:
                deltas[t, v] = 0
                last_obs = t
            else:
                deltas[t, v] = t - last_obs if t > 0 else 0

    return values, mask, deltas

In [24]:
for horizon in HORIZONS_HOURS:
    X, M, D, y, split = build_horizon_dataset(query_points, weather_df, horizon)
    test_mask = split == "test"
    M_test, y_test = M[test_mask], y[test_mask]
    fully_missing = (M_test.sum(axis=(1,2)) == 0)
    mask_sums = M_test.sum(axis=(1,2))
    corr = stats.pointbiserialr(y_test, mask_sums)
    print(f"horizon {horizon}h: fully_missing={fully_missing.sum()}/{len(M_test)}, mask-label correlation={corr.statistic:.3f} (p={corr.pvalue:.3f})")

/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:5534: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rpb, prob = pearsonr(x, y)


horizon 6h: fully_missing=0/171, mask-label correlation=nan (p=nan)


/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:5534: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rpb, prob = pearsonr(x, y)


horizon 12h: fully_missing=0/171, mask-label correlation=nan (p=nan)


/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:5534: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rpb, prob = pearsonr(x, y)


horizon 24h: fully_missing=0/171, mask-label correlation=nan (p=nan)


/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:5534: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rpb, prob = pearsonr(x, y)


horizon 48h: fully_missing=0/171, mask-label correlation=nan (p=nan)
horizon 72h: fully_missing=0/171, mask-label correlation=nan (p=nan)


/usr/local/lib/python3.12/dist-packages/scipy/stats/_stats_py.py:5534: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rpb, prob = pearsonr(x, y)


In [25]:
X, M, D, y, split = build_horizon_dataset(query_points, weather_df, 6)
test_mask = split == "test"
M_test = M[test_mask]
mask_sums = M_test.sum(axis=(1,2))
print(f"unique mask_sum values in test set: {np.unique(mask_sums)}")
print(f"min/max: {mask_sums.min()}, {mask_sums.max()}")

unique mask_sum values in test set: [192.]
min/max: 192.0, 192.0


In [26]:
grud_results = []

for horizon in HORIZONS_HOURS:
    print(f"\n=== GRU-D, horizon {horizon}h ===")
    pr_mean, pr_lo, pr_hi, n_train, n_test, pos_rate = train_grud_for_horizon(query_points, weather_df, horizon)
    grud_results.append({"arm": "grud_weather", "horizon_hours": horizon, "model": "grud",
                          "pr_auc": pr_mean, "ci_low": pr_lo, "ci_high": pr_hi,
                          "n_train": n_train, "n_test": n_test, "test_pos_rate": pos_rate,
                          "chance_level": pos_rate, "lift_over_chance": pr_mean/pos_rate})
    print(f"  PR-AUC={pr_mean:.3f} [{pr_lo:.3f},{pr_hi:.3f}]")

grud_results_df = pd.DataFrame(grud_results)
grud_results_df.to_csv("/kaggle/working/raw/phase4_grud_results.csv", index=False)
print(grud_results_df)


=== GRU-D, horizon 6h ===
    epoch 0: loss=0.7097
    epoch 10: loss=0.6702
    epoch 20: loss=0.6426
  PR-AUC=0.761 [0.679,0.837]

=== GRU-D, horizon 12h ===
    epoch 0: loss=0.7232
    epoch 10: loss=0.6534
    epoch 20: loss=0.6081
  PR-AUC=0.795 [0.716,0.863]

=== GRU-D, horizon 24h ===
    epoch 0: loss=0.7119
    epoch 10: loss=0.6692
    epoch 20: loss=0.6362
  PR-AUC=0.875 [0.795,0.939]

=== GRU-D, horizon 48h ===
    epoch 0: loss=0.7578
    epoch 10: loss=0.6499
    epoch 20: loss=0.6110
  PR-AUC=0.905 [0.854,0.945]

=== GRU-D, horizon 72h ===
    epoch 0: loss=0.7067
    epoch 10: loss=0.6594
    epoch 20: loss=0.6271
  PR-AUC=0.849 [0.761,0.918]
            arm  horizon_hours model    pr_auc    ci_low   ci_high  n_train  \
0  grud_weather              6  grud  0.761150  0.679103  0.837099      731   
1  grud_weather             12  grud  0.795229  0.716261  0.862713      731   
2  grud_weather             24  grud  0.875415  0.795439  0.939466      731   
3  grud_weather

In [27]:
xgb_weather_only = results_df[(results_df["arm"]=="weather_only") & (results_df["model"]=="xgboost")][["horizon_hours","pr_auc","lift_over_chance"]].reset_index(drop=True)
grud_compare = grud_results_df[["horizon_hours","pr_auc","lift_over_chance"]].reset_index(drop=True)

comparison = xgb_weather_only.merge(grud_compare, on="horizon_hours", suffixes=("_xgb", "_grud"))
comparison["grud_minus_xgb_lift"] = comparison["lift_over_chance_grud"] - comparison["lift_over_chance_xgb"]
print(comparison)

   horizon_hours  pr_auc_xgb  lift_over_chance_xgb  pr_auc_grud  \
0              6    0.892283              1.774190     0.761150   
1             12    0.891938              1.773505     0.795229   
2             24    0.885751              1.761202     0.875415   
3             48    0.901496              1.792510     0.904623   
4             72    0.890223              1.770095     0.848514   

   lift_over_chance_grud  grud_minus_xgb_lift  
0               1.513449            -0.260741  
1               1.581211            -0.192294  
2               1.740651            -0.020551  
3               1.798726             0.006216  
4               1.687162            -0.082933  


In [28]:
# Naive baseline: does knowing lat/lon alone (no weather) predict as well as weather features?
from sklearn.linear_model import LogisticRegression

df = cleaned_arms["weather_only"]
for horizon in HORIZONS_HOURS:
    subset = df[df["horizon_hours"] == horizon]
    train = subset[subset["split"]=="train"]
    test = subset[subset["split"]=="test"]

    # merge lat/lon back in from query_points
    train_latlon = train.merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)
    test_latlon = test.merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)

    lr = LogisticRegression(class_weight="balanced", random_state=42)
    lr.fit(train_latlon[["lat","lon"]], train_latlon["label"])
    scores = lr.predict_proba(test_latlon[["lat","lon"]])[:,1]
    pr, lo, hi = bootstrap_pr_auc(test_latlon["label"].values, scores)
    print(f"horizon {horizon}h — lat/lon only baseline: PR-AUC={pr:.3f} [{lo:.3f},{hi:.3f}]")

horizon 6h — lat/lon only baseline: PR-AUC=0.794 [0.710,0.866]
horizon 12h — lat/lon only baseline: PR-AUC=0.794 [0.710,0.866]
horizon 24h — lat/lon only baseline: PR-AUC=0.794 [0.710,0.866]
horizon 48h — lat/lon only baseline: PR-AUC=0.794 [0.710,0.866]
horizon 72h — lat/lon only baseline: PR-AUC=0.794 [0.710,0.866]


In [29]:
# The real test: weather + lat/lon vs lat/lon alone — does weather add anything?
for horizon in HORIZONS_HOURS:
    subset = df[df["horizon_hours"] == horizon]
    train = subset[subset["split"]=="train"]
    test = subset[subset["split"]=="test"]

    train_latlon = train.merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)
    test_latlon = test.merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)

    weather_feature_cols = [c for c in df.columns if c.startswith("weather_")]
    feature_cols_with_latlon = weather_feature_cols + ["lat", "lon"]

    X_train = train_latlon[feature_cols_with_latlon].fillna(train_latlon[feature_cols_with_latlon].median())
    X_test = test_latlon[feature_cols_with_latlon].fillna(train_latlon[feature_cols_with_latlon].median())

    lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
    lr.fit(X_train, train_latlon["label"])
    scores = lr.predict_proba(X_test)[:,1]
    pr, lo, hi = bootstrap_pr_auc(test_latlon["label"].values, scores)
    print(f"horizon {horizon}h — weather+latlon: PR-AUC={pr:.3f} [{lo:.3f},{hi:.3f}]")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


horizon 6h — weather+latlon: PR-AUC=0.918 [0.872,0.954]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


horizon 12h — weather+latlon: PR-AUC=0.917 [0.869,0.954]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


horizon 24h — weather+latlon: PR-AUC=0.914 [0.865,0.950]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


horizon 48h — weather+latlon: PR-AUC=0.928 [0.883,0.963]
horizon 72h — weather+latlon: PR-AUC=0.933 [0.892,0.965]


In [30]:
# Quick check: does raising max_iter change the result meaningfully?
lr_test = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)
# rerun on one horizon (e.g., 72h) to spot-check
subset = df[df["horizon_hours"] == 72]
train = subset[subset["split"]=="train"].merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)
test = subset[subset["split"]=="test"].merge(query_points[["lat","lon"]], left_on="query_point_idx", right_index=True)
feature_cols_with_latlon = [c for c in df.columns if c.startswith("weather_")] + ["lat", "lon"]
X_train = train[feature_cols_with_latlon].fillna(train[feature_cols_with_latlon].median())
X_test = test[feature_cols_with_latlon].fillna(train[feature_cols_with_latlon].median())
lr_test.fit(X_train, train["label"])
scores = lr_test.predict_proba(X_test)[:,1]
pr, lo, hi = bootstrap_pr_auc(test["label"].values, scores)
print(f"72h with max_iter=5000: PR-AUC={pr:.3f} [{lo:.3f},{hi:.3f}] (compare to 0.933 [0.892,0.965])")

72h with max_iter=5000: PR-AUC=0.933 [0.892,0.965] (compare to 0.933 [0.892,0.965])
